# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

This notebook simulates role-based column masking and dbt-style metric calculations (30-day readmissions) using PySpark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, lit, count, when

spark = SparkSession.builder \
    .appName("Day09_Gold_Masking") \
    .getOrCreate()

print("Spark session initialized.")

In [ ]:
# 1. Simulate dbt Gold Marts (30-Day Readmission)
data = [
    ("P001", "2023-01-01", 32020), # Index admission
    ("P001", "2023-01-15", 32020), # Readmission within 30 days
    ("P002", "2023-02-01", 32020), # Index admission
    ("P002", "2023-04-01", 32020), # Not a 30-day readmission
]
columns = ["person_id", "condition_start_date", "condition_type_concept_id"]
condition_df = spark.createDataFrame(data, columns)

condition_df.createOrReplaceTempView("condition_data")

readmissions_sql = """
SELECT 
    a.person_id,
    COUNT(*) as thirty_day_readmission_count
FROM condition_data a
JOIN condition_data b 
    ON a.person_id = b.person_id
    AND b.condition_start_date > a.condition_start_date
    AND datediff(b.condition_start_date, a.condition_start_date) <= 30
GROUP BY a.person_id
"""

gold_readmissions_df = spark.sql(readmissions_sql)
print("Gold Mart: 30-Day Readmissions")
gold_readmissions_df.show()

In [ ]:
# 2. Simulate Dynamic Data Masking based on Role
patient_data = [("P001", "123-45-6789", "1980-05-15")]
patient_df = spark.createDataFrame(patient_data, ["person_id", "ssn", "dob"])

def apply_masking(df, role):
    if role == "ADMIN":
        return df # No masking
    else:
        return df \
            .withColumn("ssn", lit("***-**-****")) \
            .withColumn("dob", col("dob").substr(1, 4)) # Only show year

print("View as ADMIN:")
apply_masking(patient_df, "ADMIN").show()

print("View as ANALYST:")
apply_masking(patient_df, "ANALYST").show()
